#Phase 3 的核心：Qwen3-0.6B 的 SFT + 标准 LoRA 微调
用我们准备好的物流样本，让 Qwen3-0.6B 学会更稳定地输出正确答案。这是项目第一次真正训练模型。

整个阶段按下面顺序推进，每次只做一步：

| 步骤 | 做什么 | 通俗理解 |
|---|---|---|
| **1. 检查环境** | 确认 GPU、显存和依赖版本 | 先检查训练设备能不能正常用 |
| **2. 准备训练输入** | 读取 Train/Val，套用聊天模板、转换为 token，检查长度和答案范围 | 把问答整理成模型能学习的格式 |
| **3. 加载模型并配置 LoRA** | 加载 Qwen，冻结原参数，添加可训练的 LoRA 参数 | 保留原有知识，通过少量新增参数学习物流任务 |
| **4. 设置训练方案** | 确定学习率、训练轮数、批量大小和 Loss | 决定每次学多少、学多快、学几遍 |
| **5. 训练与验证** | 执行 SFT，记录训练及验证 Loss，并评估任务准确率 | 看模型有没有学会，以及是否出现过拟合 |
| **6. 保存训练成果** | 保存 LoRA Adapter、分词器、配置和日志 | 让训练结果可以重新加载和复现 |

这里 **SFT 和 LoRA 的关系**是：

| 概念 | 在我们项目中的作用 |
|---|---|
| **SFT：监督微调** | 用“物流问题＋标准答案”教模型 |
| **LoRA：参数高效微调方法** | 决定训练哪些参数，减少显存和训练开销 |

我们主要希望改善：

- **Task A**：保持严格 JSON 输出，提高字段提取和缺失值 `null` 的准确率。
- **Task B**：减少一律回答 `delayed` 的现象，检查模型能否正确区分两类。

**Phase 3 结束应拿到：可加载的 LoRA Adapter、训练配置、训练与验证记录。提升多少要靠实测；**

Phase 4 再做系统对比，测试集继续保留。

## 1.检查训练环境

本阶段使用 Phase 1 准备的物流指令数据，微调 Qwen3-0.6B。

- Task A：物流信息提取，重点改善字段提取和缺失值 null 的处理。
- Task B：交付延误判断，检验模型能否学会遵循时间比较规则。
- 使用标准 LoRA，只训练少量新增参数，冻结基座模型参数。
- 使用训练集进行训练，验证集用于选择方案。
- 测试集继续保留，方案确定后再进行最终评测。

本步骤仅检查 GPU 和依赖，不开始训练。

In [ ]:
# 第 1 步：检查 Colab GPU 和主要依赖版本
# 此步骤不安装或升级软件，避免直接改变已有环境。

import sys
from importlib.metadata import version, PackageNotFoundError

print("===== Python 与依赖版本 =====")
print("Python:", sys.version.split()[0])

packages = [
    "torch",
    "transformers",
    "datasets",
    "accelerate",
    "peft",
    "trl",
]

missing_packages = []

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: 未安装")
        missing_packages.append(package)

print("\n===== GPU 状态 =====")

try:
    import torch

    gpu_available = torch.cuda.is_available()
    print("CUDA 是否可用:", gpu_available)
    print("PyTorch 对应 CUDA 版本:", torch.version.cuda)

    if gpu_available:
        # 查看当前 GPU 的名称、总显存和当前可用显存
        device = torch.cuda.current_device()
        gpu = torch.cuda.get_device_properties(device)
        free_memory, total_memory = torch.cuda.mem_get_info(device)

        print("GPU 名称:", gpu.name)
        print(f"GPU 总显存: {total_memory / 1024**3:.2f} GB")
        print(f"当前可用显存: {free_memory / 1024**3:.2f} GB")
    else:
        print("请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU。")

except ImportError:
    gpu_available = False
    print("PyTorch 未安装，暂时无法检查 GPU。")

print("\n===== 检查结论 =====")

if missing_packages:
    print("缺少依赖:", ", ".join(missing_packages))

if gpu_available and not missing_packages:
    print("GPU 和主要依赖已检测到；下一步确认兼容性并准备训练数据。")
else:
    print("需要先处理上述环境问题，再继续训练准备。")

In [ ]:
# 补装 TRL，并保留当前主要依赖版本。
# 如果出现版本冲突，保留完整报错，不要继续升级或卸载其他包。

%pip install trl "torch==2.11.0" "transformers==5.21.0" "datasets==4.8.5" "accelerate==1.15.0" "peft==0.21.2"

In [ ]:
# 移除标准 LoRA 不需要的可选量化库
%pip uninstall -y torchao

In [ ]:
# 验证 SFT 和标准 LoRA 所需组件是否能正常导入
# 不加载模型，不开始训练

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

print("SFTTrainer / SFTConfig：导入成功")
print("LoraConfig / get_peft_model：导入成功")
print("Phase 3 第 1 步完成，可以准备训练数据。")

## 2. 读取并检查训练数据

使用 Phase 1 正式版数据：
- train.jsonl：1,600 条，用于训练。
- val.jsonl：200 条，用于验证。

检查 JSONL 能否正常读取，以及 messages 中的问题和标准答案是否完整。
本步骤不修改原始数据，不读取 test.jsonl。

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

# 上传 Phase 1 正式版 ZIP 数据包
print("请上传 FleetMind_LLM_Phase1_Dataset.zip")
uploaded = files.upload()

zip_names = [
    name for name in uploaded
    if name.lower().endswith(".zip")
]
assert len(zip_names) == 1, "请一次上传一个 ZIP 数据包"

# 只提取 train.jsonl 和 val.jsonl，兼容 ZIP 内有子文件夹的情况
with zipfile.ZipFile(zip_names[0], "r") as archive:
    selected = {}

    # 先检查文件是否唯一，避免误选不同版本
    for filename in ["train.jsonl", "val.jsonl"]:
        matches = [
            item for item in archive.infolist()
            if not item.is_dir()
            and Path(item.filename).name == filename
        ]
        assert len(matches) == 1, (
            f"{filename} 找到 {len(matches)} 份，请确认 ZIP 是正式版数据包"
        )
        selected[filename] = matches[0]

    # 固定写入 /content，不使用压缩包中的目录路径
    for filename, item in selected.items():
        output_path = Path("/content") / filename
        output_path.write_bytes(archive.read(item))
        print(f"已提取：{output_path}")

print("\n提取完成，接下来运行之前的读取检查代码。")

In [ ]:
import json
from pathlib import Path

# 如果文件在其他文件夹，只需修改这里
DATA_DIR = Path("/content")

train_path = DATA_DIR / "train.jsonl"
val_path = DATA_DIR / "val.jsonl"

# 缺少文件时，上传正式版 train.jsonl 和 val.jsonl
if not (train_path.is_file() and val_path.is_file()):
    from google.colab import files
    print("请选择正式版 train.jsonl 和 val.jsonl")
    files.upload()

assert train_path.is_file(), f"未找到：{train_path}"
assert val_path.is_file(), f"未找到：{val_path}"


def read_and_check(path, expected_count):
    """逐行读取 JSON，并检查单轮问答结构。"""
    records = []

    with path.open("r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            assert isinstance(record, dict), f"{path.name} 第 {line_number} 行不是对象"

            messages = record.get("messages")
            assert isinstance(messages, list) and messages, (
                f"{path.name} 第 {line_number} 行缺少有效 messages"
            )

            for message in messages:
                assert isinstance(message, dict), "消息必须为对象"
                assert isinstance(message.get("content"), str), "消息内容必须为文本"
                assert message["content"].strip(), "消息内容不能为空"

            # 当前数据应为：可选 system + user + assistant
            roles = [message.get("role") for message in messages]
            assert roles in (
                ["user", "assistant"],
                ["system", "user", "assistant"],
            ), f"{path.name} 第 {line_number} 行角色顺序异常：{roles}"

            records.append(record)

    assert len(records) == expected_count, (
        f"{path.name} 应有 {expected_count} 条，实际 {len(records)} 条。"
        "请确认使用的是正式版数据。"
    )

    print(f"{path.name}：{len(records)} 条，问答格式检查通过")
    return records


# 只读取训练集和验证集
train_records = read_and_check(train_path, 1600)
val_records = read_and_check(val_path, 200)

# 查看一条训练样本，供下一步确定聊天模板和答案处理方式
print("\n===== 第一条训练样本 =====")
print(json.dumps(train_records[0], ensure_ascii=False, indent=2))

print("\n读取完成；测试集未读取，原始文件未修改。")

## 3. 聊天模板与 Token 长度检查

- 加载 Qwen3-0.6B 的分词器，暂不加载模型权重。
- 使用模型自带的聊天模板，采用非思考模式。
- 统计训练集和验证集的完整问答长度。
- 本步骤不截断样本，根据统计结果再确定最大序列长度。
- sample_id 等元数据用于追踪，不放入模型输入。

In [ ]:
from transformers import AutoTokenizer
import pandas as pd

MODEL_ID = "Qwen/Qwen3-0.6B"

# 只加载分词器，不加载模型权重
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def format_conversation(record):
    # 只将 messages 放入聊天模板，保留完整的标准答案
    return tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )


# 生成完整问答文本，供后续训练准备使用
train_texts = [format_conversation(r) for r in train_records]
val_texts = [format_conversation(r) for r in val_records]


def summarize_lengths(texts, split):
    # 模板已包含特殊标记，因此不重复添加
    # 暂不截断、不填充，查看样本的真实长度
    encoded = tokenizer(
        texts,
        add_special_tokens=False,
        truncation=False,
        padding=False,
    )
    lengths = [len(ids) for ids in encoded["input_ids"]]
    series = pd.Series(lengths)

    return {
        "数据集": split,
        "样本数": len(lengths),
        "最短": min(lengths),
        "中位数": int(series.median()),
        "P95": round(series.quantile(0.95), 1),
        "最长": max(lengths),
        "超过512": sum(n > 512 for n in lengths),
        "超过1024": sum(n > 1024 for n in lengths),
    }


summary = pd.DataFrame([
    summarize_lengths(train_texts, "Train"),
    summarize_lengths(val_texts, "Validation"),
])

print("===== 完整问答 Token 长度 =====")
display(summary)

print("\n===== 第一条训练样本的聊天模板 =====")
print(train_texts[0])

print("\n检查完成：未截断样本，未读取测试集。")

## 4. 准备训练输入与答案标签

### 目的：
让模型根据物流记录学习输出标准答案，并明确哪些内容参与 Loss 计算。

- 最大序列长度设为 512，不截断当前样本。
- 系统提示、用户问题和助手回答前缀的标签设为 -100，不计算 Loss。
- 对助手标准答案及回答结束标记计算 Loss。
- 检查问题与答案的边界，避免模型学错位置。
- 本步骤仍未开始训练。
### 本步产出
得到包含 `input_ids`、`attention_mask` 和 `labels` 的训练集与验证集。

完成后，数据具备训练所需的输入和标签；本步骤尚未更新模型参数。

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512


def prepare_sample(record):
    # 完整问答：包含标准答案
    full_text = format_conversation(record)

    # 推理时的输入：不包含标准答案，以助手回答前缀结束
    prompt_text = tokenizer.apply_chat_template(
        record["messages"][:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    full_ids = tokenizer(
        full_text, add_special_tokens=False
    )["input_ids"]

    prompt_ids = tokenizer(
        prompt_text, add_special_tokens=False
    )["input_ids"]

    # 验证 token 边界，不能仅凭字符长度猜答案的位置
    assert full_ids[:len(prompt_ids)] == prompt_ids, (
        f"{record['sample_id']}：问题与答案边界不匹配"
    )
    assert len(full_ids) <= MAX_LENGTH, (
        f"{record['sample_id']}：超过最大长度，请先检查，不自动截断"
    )

    # -100 是交叉熵损失默认忽略的标签
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    assert any(label != -100 for label in labels), "没有可学习的答案"

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }


# 暂不 padding，后续组批时再按批次填充，节省显存
train_dataset = Dataset.from_list([
    prepare_sample(record) for record in train_records
])
val_dataset = Dataset.from_list([
    prepare_sample(record) for record in val_records
])

print("===== 训练输入准备完成 =====")
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("最大序列长度:", MAX_LENGTH)

# 查看实际参与 Loss 计算的内容
example = train_dataset[0]
answer_ids = [
    token_id
    for token_id, label in zip(example["input_ids"], example["labels"])
    if label != -100
]

print("\n===== 第一条样本的标签检查 =====")
print("完整输入 token 数:", len(example["input_ids"]))
print("忽略的 token 数:", example["labels"].count(-100))
print("参与 Loss 的 token 数:", len(answer_ids))

print("\n===== 实际学习的答案内容 =====")
print(tokenizer.decode(answer_ids, skip_special_tokens=False))

print("\n全部样本边界检查通过；尚未开始训练。")

## 5. 加载 Qwen 并配置标准 LoRA

### 目的
保留 Qwen 的原有参数，通过少量新增参数学习物流任务，降低训练显存和参数更新开销。

### 配置与作用
| 配置 | 设置 | 作用 |
|---|---|---|
| 基座模型 | Qwen3-0.6B | 提供已有语言能力 |
| 基座加载精度 | float16 | 减少模型权重的显存占用 |
| 微调方法 | 标准 LoRA | 冻结基座，仅更新 LoRA 参数 |
| 目标模块 | q_proj、k_proj、v_proj、o_proj | 调整注意力层中的信息处理 |
| Rank | 16 | 控制新增参数规模，作为首轮实验配置 |
| Alpha | 32 | 标准 LoRA 的缩放系数为 alpha/r，即 2 |
| Dropout | 0.05 | 在 LoRA 分支加入正则化，帮助抑制过拟合 |

### 本步产出
得到已添加 LoRA 的模型，并确认可训练参数数量及占比。
这些是首轮实验设置，实际效果由后续验证结果判断。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, set_seed
from peft import LoraConfig, get_peft_model, TaskType

assert torch.cuda.is_available(), "请先启用 Colab GPU"
set_seed(42)

# 仅首次运行此单元：避免对已添加 LoRA 的模型重复添加
# 首次下载模型权重需要等待，后续可复用缓存
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")

# 训练阶段不使用生成缓存
model.config.use_cache = False

# 设置填充 token；后续填充位置的标签仍须设为 -100
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
model.config.pad_token_id = tokenizer.pad_token_id

# 标准 LoRA：不量化基座，不启用其他 LoRA 变体
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    use_rslora=False,
    use_dora=False,
)

# 添加 LoRA；PEFT 会冻结基座参数
model = get_peft_model(model, lora_config)

print("===== 可训练参数 =====")
model.print_trainable_parameters()

# 确认没有意外开放基座参数
trainable = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

assert trainable, "没有可训练参数"
assert all("lora_" in name for name, _ in trainable), (
    "检测到 LoRA 以外的可训练参数，请检查配置"
)

print("\n===== 模型状态 =====")
print("设备:", next(model.parameters()).device)
print("基座权重精度:", model.get_input_embeddings().weight.dtype)
print("可训练参数精度:", sorted({str(p.dtype) for _, p in trainable}))
print("仅 LoRA 参数可训练：检查通过")
print(f"当前张量占用显存：{torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("\n模型准备完成，尚未开始训练。")

## 6. 设置训练方式

### 这一步做什么？
确定模型学几遍、每次学多少，以及多久检查一次效果。

### 训练设置
| 设置 | 数值 | 通俗解释 |
|---|---|---|
| 训练轮数 | 3 轮 | 将训练集完整学习 3 遍 |
| 学习率 | 0.0001 | 控制每次调整参数的幅度 |
| 每批样本数 | 2 条 | 每次处理 2 条，降低显存压力 |
| 梯度累积 | 8 次 | 累积处理 16 条样本后，再更新一次参数 |
| 混合精度 | FP16 | 减少训练时的显存和计算开销 |
| 验证与保存 | 每轮一次 | 每学完一轮就检查效果、保存进度 |

### 为什么要检查填充？
同一批中的问答长度不同，需要把较短的样本补齐。

补齐的位置只是为了方便计算，不是标准答案，因此不能参与 Loss。代码会检查：
- 原来的答案标签是否保留。
- 补齐的位置是否已设为忽略。

### 完成后得到什么？
得到训练配置，并确认不同长度的样本可以正确组成一批。

本步骤只做好训练准备，还没有开始训练。

In [ ]:
from trl import SFTConfig
from transformers import DataCollatorForSeq2Seq

# 保留已有 labels，只负责把同一批次的输入和标签填充到相同长度
# 不传 model，避免引入 encoder-decoder 专用处理
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# 首轮训练配置：暂不创建 Trainer，不启动训练
training_args = SFTConfig(
    output_dir="/content/fleetmind_phase3_lora",
    num_train_epochs=3,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    optim="adamw_torch",
    lr_scheduler_type="linear",
    warmup_steps=30,   # 前 30 次参数更新逐渐提高学习率
    max_grad_norm=1.0,
    weight_decay=0.01,

    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    max_length=MAX_LENGTH,
    packing=False,
    # 数据已分词并设置答案标签，跳过再次加工
    dataset_kwargs={"skip_dataset_prepare": True},
    report_to="none",
    seed=42,
    data_seed=42,
)

# 选取长短不同的两条样本，实际测试动态填充
lengths = [len(row["input_ids"]) for row in train_dataset]
indices = [
    min(range(len(lengths)), key=lengths.__getitem__),
    max(range(len(lengths)), key=lengths.__getitem__),
]
features = [train_dataset[i] for i in indices]
batch = data_collator(features)

# 检查原始标签未被覆盖
for row_index, feature in enumerate(features):
    size = len(feature["labels"])
    assert batch["labels"][row_index, :size].tolist() == feature["labels"]

# 检查所有填充位置都不参与 Loss
padding_positions = batch["attention_mask"] == 0
assert padding_positions.any(), "本次检查未产生填充位置"
assert (batch["labels"][padding_positions] == -100).all()

print("===== 配置与组批检查 =====")
print("训练轮数:", training_args.num_train_epochs)
print("学习率:", training_args.learning_rate)
print("单卡有效批量: 2 × 8 = 16")
print("预计每轮参数更新: 100 次；3 轮合计: 300 次")
print("批次形状:", tuple(batch["input_ids"].shape))
print("原始答案标签保留：通过")
print("填充位置忽略 Loss：通过")
print("\n配置完成，尚未开始训练。")

## 7. 训练前试跑

### 目的
正式训练前，用一个小批次检查计算流程是否正常。

| 检查 | 作用 |
|---|---|
| 前向传播 | 检查模型能否计算 Loss |
| 反向传播 | 检查 LoRA 参数能否获得梯度 |
| 数值检查 | 确认 Loss 和梯度没有出现 NaN 或无穷大 |
| 显存记录 | 观察当前批次的显存开销 |

本次不执行参数更新，检查结束后清空梯度。
试跑 Loss 只用于排查问题，不代表完整训练集或验证集成绩。

In [ ]:
import gc
import torch
from transformers import set_seed

# 与正式训练配置一致，启用梯度检查点以节省显存
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
model.config.use_cache = False
model.train()
model.zero_grad(set_to_none=True)

torch.cuda.reset_peak_memory_stats()

# 使用上一步选出的长短两条样本，检查真实批次计算
smoke_batch = {
    key: value.to("cuda")
    for key, value in data_collator(features).items()
}

try:
    # 前向传播：计算只针对答案部分的交叉熵 Loss
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        smoke_output = model(**smoke_batch)
        smoke_loss = smoke_output.loss

    assert torch.isfinite(smoke_loss).item(), "Loss 出现 NaN 或无穷大"
    loss_value = smoke_loss.item()

    # 反向传播：只计算梯度，不执行 optimizer.step()
    smoke_loss.backward()

    grad_count = 0
    nonzero_grad_count = 0

    for name, parameter in model.named_parameters():
        if parameter.requires_grad and parameter.grad is not None:
            grad_count += 1
            assert torch.isfinite(parameter.grad).all().item(), (
                f"梯度出现异常：{name}"
            )
            if torch.count_nonzero(parameter.grad).item() > 0:
                nonzero_grad_count += 1

    assert nonzero_grad_count > 0, "未检测到非零梯度"

    print("===== 训练前试跑 =====")
    print(f"当前批次 Loss：{loss_value:.4f}")
    print("获得梯度的参数张量数：", grad_count)
    print("梯度非零的参数张量数：", nonzero_grad_count)
    print("Loss 和梯度数值检查：通过")
    print(
        f"试跑峰值张量显存："
        f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
    )
    print("\n试跑通过，未执行参数更新。")

finally:
    # 无论检查是否成功，都清理梯度和临时计算结果
    model.zero_grad(set_to_none=True)
    del smoke_batch
    if "smoke_output" in globals():
        del smoke_output
    if "smoke_loss" in globals():
        del smoke_loss
    gc.collect()
    torch.cuda.empty_cache()
    set_seed(42)

试跑通过，可以开始正式训练了。

112 个参数张量首次梯度非零是正常的：标准 LoRA 初始化时，一组矩阵为零，会使另一组矩阵的首次梯度为零。当前检查没有发现数值异常。

另外，0.1235 是两条样本在提供标准答案上下文时算出的 Loss，不能换算成业务准确率。

## 8. 正式进行 SFT + LoRA 训练

### 这一步做什么？
使用 1,600 条训练样本更新 LoRA 参数，共训练 3 轮。

| 操作 | 作用 |
|---|---|
| 每 10 次参数更新记录训练 Loss | 观察模型是否在学习 |
| 每轮使用 200 条验证样本评估 | 检查对未参与训练样本的表现 |
| 每轮保存检查点 | 保留训练进度 |
| 按验证 Loss 选择最佳检查点 | 避免直接使用最后一轮模型 |
| 保存最佳 LoRA 和训练日志 | 支持后续加载、评测和复现 |

### 如何理解输出？
- Loss：模型预测标准答案 token 时的误差，通常越低越好。
- mean_token_accuracy：在给定标准答案前文时，预测下一个 token 的准确率。
- 业务准确率：需要模型独立生成答案后另外评分，不能用 token 准确率代替。

本步不使用测试集。训练完成后，再检查 Task A 和 Task B 的实际生成表现。

In [ ]:
import time
from pathlib import Path
from trl import SFTTrainer
from transformers import set_seed

set_seed(42)

# 复用已有模型、分词器、训练配置和带答案标签的数据
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    # 模型已经添加 LoRA，此处不要再传 peft_config
)

print("===== 开始正式训练 =====")
print("训练样本：", len(train_dataset))
print("验证样本：", len(val_dataset))
print("训练轮数：", training_args.num_train_epochs)
print("Warmup 更新次数：", training_args.warmup_steps)

torch.cuda.reset_peak_memory_stats()
start_time = time.perf_counter()

# 正式训练：从这里开始更新 LoRA 参数
train_result = trainer.train()

elapsed_minutes = (time.perf_counter() - start_time) / 60

# 配置已要求训练结束后恢复验证 Loss 最低的检查点
# 将恢复后的最佳 LoRA 单独保存
adapter_dir = Path(training_args.output_dir) / "best_adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

# 保存指标和完整 Trainer 状态，其中包括训练/验证日志
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

print("\n===== 训练完成 =====")
print(f"实际训练耗时：{elapsed_minutes:.2f} 分钟")
print("实际参数更新次数：", trainer.state.global_step)
print("最佳检查点：", trainer.state.best_model_checkpoint)
print("最佳验证 Loss：", trainer.state.best_metric)
print("最佳 LoRA 保存位置：", adapter_dir)
print(
    f"训练峰值张量显存："
    f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GB"
)

正式训练完成，最佳模型来自第 2 轮，已经保存。
- 训练耗时：16.62 分钟，共 300 次参数更新。
- 最佳检查点：`checkpoint-200`，即第 2 轮。
- 第 3 轮没有继续改善验证 Loss，目前没有必要增加轮数；这点微小波动还不足以证明明显过拟合。

**这些结果说明模型很容易预测标准答案中的 token，但尚不能宣称物流任务准确率达到 99.993%。**  独立生成时没有标准答案前文，需要另外验证。

## 9. 抽查微调后的独立回答

### 目的
检查模型在看不到标准答案时，能否根据物流记录独立生成正确回答。

### 方法
- 使用已恢复的最佳 LoRA 模型。
- 从验证集选取 Task A、Task B 各两条。
- 输入只包含系统提示和用户问题，不包含标准答案。
- 并排查看模型回答和标准答案。

本步骤只做初步抽查，不计算正式准确率，也不使用测试集。

In [ ]:
import torch

# Trainer 已在训练结束时恢复最佳检查点
model = trainer.model
model.eval()
model.gradient_checkpointing_disable()

# A、B 各选两条验证样本
preview_records = []
for task in ["A", "B"]:
    candidates = [r for r in val_records if r["task"] == task]
    preview_records.extend(candidates[:2])

for record in preview_records:
    # 不传入 assistant 标准答案
    prompt = tokenizer.apply_chat_template(
        record["messages"][:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 只解码新生成的回答，不把输入问题算进去
    answer_ids = generated[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    print(f"\n===== {record['sample_id']} =====")
    print("模型回答：", answer)
    print("标准答案：", record["messages"][-1]["content"])

##10. 全量验证集生成评测

### 目的
检查微调后的模型能否独立完成物流任务，得到业务指标。

| 任务 | 记录指标 | 检查重点 |
|---|---|---|
| Task A | JSON 有效率、结构正确率、字段准确率 | 输出是否合法、字段是否正确 |
| Task A 缺失信息 | null 准确率 | 是否把未提供的信息正确保留为 null |
| Task B | Accuracy、Macro-F1、on_time Recall | 是否正确区分两类，避免只预测 delayed |

### 评测方法
使用全部 200 条验证样本，输入不包含标准答案。
保持本次抽查的提示词处理和生成设置，保存逐条预测以便分析错误。

本次结果先单独记录。与 Phase 2 比较前，再核对双方评测设置是否一致。

In [ ]:
import json
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

model.eval()

# 拒绝 JSON 标准之外的 NaN、Infinity 等值
def reject_constant(value):
    raise ValueError(f"非法 JSON 常量：{value}")

def same_value(predicted, expected):
    # 避免把 True 和 1 误判为相同
    if expected is None:
        return predicted is None
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        return (
            isinstance(predicted, (int, float))
            and not isinstance(predicted, bool)
            and predicted == expected
        )
    return type(predicted) is type(expected) and predicted == expected

counts = Counter()
b_pairs = []
predictions = []

for record in tqdm(val_records, desc="验证集独立生成"):
    task = record["task"]
    assert task in ["A", "B"]

    prompt = tokenizer.apply_chat_template(
        record["messages"][:-1],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to("cuda")

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    answer_ids = generated[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)
    expected = json.loads(record["messages"][-1]["content"])

    # 直接解析原回答，不剥离代码围栏或提取其中的 JSON
    parsed = None
    valid_json = False
    try:
        parsed = json.loads(answer, parse_constant=reject_constant)
        valid_json = True
    except (ValueError, TypeError):
        pass

    counts[f"{task}_total"] += 1

    if task == "A":
        counts["A_json_valid"] += int(valid_json)
        is_object = isinstance(parsed, dict)
        correct_keys = is_object and set(parsed) == set(expected)

        # 结构检查键名及各字段类型，允许缺失信息使用 null
        def valid_type(value, reference):
            if value is None:
                return True
            if reference is None:
                return isinstance(value, str)  # 当前缺失字段为时间
            if isinstance(reference, (int, float)):
                return isinstance(value, (int, float)) and not isinstance(value, bool)
            return type(value) is type(reference)

        structure_ok = correct_keys and all(
            valid_type(parsed[key], expected[key]) for key in expected
        )
        counts["A_structure"] += int(structure_ok)

        for key, reference in expected.items():
            correct = (
                is_object and key in parsed
                and same_value(parsed[key], reference)
            )
            counts["A_fields_total"] += 1
            counts["A_fields_correct"] += int(correct)

            if reference is None:
                counts["A_null_total"] += 1
                counts["A_null_correct"] += int(correct)

    else:
        # 格式错误、额外字段或非法标签均记为错误预测
        predicted_label = "__invalid__"
        if (
            isinstance(parsed, dict)
            and set(parsed) == {"delay_label"}
            and parsed["delay_label"] in ["delayed", "on_time"]
        ):
            predicted_label = parsed["delay_label"]

        b_pairs.append((expected["delay_label"], predicted_label))

    predictions.append({
        "sample_id": record["sample_id"],
        "task": task,
        "prediction_text": answer,
        "expected": expected,
        "json_valid": valid_json,
    })

# 手动计算两类指标，非法预测也会计入漏判
class_metrics = {}
for label in ["delayed", "on_time"]:
    tp = sum(y == label and p == label for y, p in b_pairs)
    fp = sum(y != label and p == label for y, p in b_pairs)
    fn = sum(y == label and p != label for y, p in b_pairs)
    class_metrics[label] = {
        "f1": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0,
        "recall": tp / (tp + fn) if tp + fn else 0,
    }

def ratio(numerator, denominator):
    return numerator / denominator if denominator else None

metrics = {
    "A_json_validity": ratio(counts["A_json_valid"], counts["A_total"]),
    "A_structure_accuracy": ratio(counts["A_structure"], counts["A_total"]),
    "A_field_accuracy": ratio(counts["A_fields_correct"], counts["A_fields_total"]),
    "A_null_accuracy": ratio(counts["A_null_correct"], counts["A_null_total"]),
    "B_accuracy": ratio(sum(y == p for y, p in b_pairs), len(b_pairs)),
    "B_macro_f1": sum(m["f1"] for m in class_metrics.values()) / 2,
    "B_on_time_recall": class_metrics["on_time"]["recall"],
}

# 保存逐条结果和汇总
result_dir = Path(training_args.output_dir) / "validation_results"
result_dir.mkdir(parents=True, exist_ok=True)

with (result_dir / "predictions.jsonl").open("w", encoding="utf-8") as file:
    for item in predictions:
        file.write(json.dumps(item, ensure_ascii=False) + "\n")

summary = {
    "metrics": metrics,
    "counts": dict(counts),
    "B_prediction_counts": dict(Counter(p for _, p in b_pairs)),
    "checkpoint": trainer.state.best_model_checkpoint,
}
(result_dir / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\n===== 全量验证结果 =====")
for name, value in metrics.items():
    print(f"{name}: {value:.4%}" if value is not None else f"{name}: 无适用样本")
print("计数：", dict(counts))
print("Task B 预测分布：", summary["B_prediction_counts"])
print("结果保存位置：", result_dir)

全量验证完成：Task A 全部正确，Task B 正确 99/100。首轮 SFT + LoRA 已取得有效结果

Task B 的结果对应：67 条延误全部识别正确，33 条准时中有 1 条被误判为延误。模型已经摆脱之前“一律预测 delayed”的问题。

目前可以得出：**模型在这批验证数据上，能稳定提取字段、保留缺失值，并基本遵循时间判断规则**。这些是验证集成绩；测试集尚未使用，对真实业务和新表达方式的泛化能力仍需后续检查。与 Phase 2 的提升幅度，也要在统一评测设置后正式计算。

下一步先看 Task B 唯一的错误案例，暂时不调整训练参数。

## 11. 查看 Task B 错误案例

### 目的
找出唯一判断错误的样本，检查错误是否与时间比较、输入表达或输出格式有关。

### 本步产出
记录错误样本的原始问题、模型回答和标准答案，为后续分析提供证据。

In [ ]:
# 使用刚才保存的内存结果，不重新生成回答
val_lookup = {r["sample_id"]: r for r in val_records}
error_count = 0

for item in predictions:
    if item["task"] != "B":
        continue

    try:
        parsed = json.loads(
            item["prediction_text"],
            parse_constant=reject_constant,
        )
    except (ValueError, TypeError):
        parsed = None

    if parsed != item["expected"]:
        error_count += 1
        record = val_lookup[item["sample_id"]]

        print(f"\n===== 错误案例：{item['sample_id']} =====")
        print("原始问题：", record["messages"][-2]["content"])
        print("模型回答：", item["prediction_text"])
        print("标准答案：", item["expected"])

print("\nTask B 错误数：", error_count)

**这条记录实际提前约 0.581 秒完成，正确标签是 `on_time`。模型在极接近的时间比较上出现了错误。**


两者日期、小时、分钟完全相同，需要进一步比较秒数。**这说明模型仍存在精细时间比较的弱点**；仅凭这一条，无法确定它具体忽略了哪部分数字，也不能归因于训练轮数不足。


**当前阶段结论：Task A 验证集全部正确；Task B 为 99%，剩余错误出现在不足 1 秒的时间边界**。先保留这版最佳 LoRA 和结果，后续进入统一条件下的正式对比。

## 12. 备份模型与实验结果

### 目的
保存本轮训练成果，方便以后恢复模型、复查结果和整理报告。

| 保存内容 | 用途 |
|---|---|
| 最佳 LoRA Adapter 与分词器 | 重新加载微调模型 |
| 训练日志 | 查看训练和验证过程 |
| 验证集预测与指标 | 复查成绩和错误案例 |
| 实验配置与数据文件指纹 | 记录本次使用的环境、参数和数据版本 |

本步将以上内容打包为 ZIP 并下载到电脑。

In [ ]:
import json
import hashlib
import zipfile
from pathlib import Path
from importlib.metadata import version
from google.colab import files

root = Path("/content/fleetmind_phase3_lora")
adapter_dir = root / "best_adapter"
results_dir = root / "validation_results"

# 确认关键成果存在，避免下载空包
for required in [
    adapter_dir / "adapter_config.json",
    adapter_dir / "adapter_model.safetensors",
    results_dir / "predictions.jsonl",
    results_dir / "summary.json",
    root / "trainer_state.json",
]:
    assert required.is_file(), f"缺少文件：{required}"


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# 记录关键配置和数据指纹，不需要将训练数据重复放入压缩包
manifest = {
    "project": "FleetMind-LLM",
    "base_model": MODEL_ID,
    "base_model_revision": getattr(model.config, "_commit_hash", None),
    "seed": 42,
    "epochs": 3,
    "learning_rate": 1e-4,
    "batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_steps": 30,
    "max_length": MAX_LENGTH,
    "lora_rank": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_loss": trainer.state.best_metric,
    "actual_optimizer_steps": trainer.state.global_step,
    "dataset_sha256": {
        "train.jsonl": sha256_file("/content/train.jsonl"),
        "val.jsonl": sha256_file("/content/val.jsonl"),
    },
    "packages": {
        name: version(name)
        for name in [
            "torch", "transformers", "datasets",
            "accelerate", "peft", "trl"
        ]
    },
    "test_set_used": False,
}

manifest_path = root / "experiment_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# 只打包最佳 Adapter、验证结果和日志，减少重复检查点占用
zip_path = Path("/content/FleetMind_LLM_Phase3_Results.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for folder in [adapter_dir, results_dir]:
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, arcname=str(path.relative_to(root)))

    for filename in [
        "experiment_manifest.json",
        "trainer_state.json",
        "train_results.json",
    ]:
        path = root / filename
        if path.is_file():
            archive.write(path, arcname=filename)

print("备份文件：", zip_path.name)
print(f"文件大小：{zip_path.stat().st_size / 1024**2:.2f} MB")
print("请保留 Phase 1 原始数据包，恢复实验时仍需使用。")

files.download(str(zip_path))

## 总结

| 项目 | 阶段结果 |
|---|---|
| 微调方式 | Qwen3-0.6B + SFT + 标准 LoRA |
| 可训练参数 | 458.75 万，占 0.7638% |
| 训练 | 3 轮、300 次更新，约 16.62 分钟 |
| 最佳模型 | 第 2 轮，验证 Loss 为 0.000562 |
| Task A | JSON、结构、字段、`null` 准确率均为 100% |
| Task B | Accuracy 99%，Macro-F1 98.86% |
| 唯一错误 | 将提前约 0.581 秒的准时记录判为延误 |
| 成果保存 | 最佳 LoRA、日志、验证结果及配置已备份 |
| 测试集 | 尚未使用 |

**下一阶段是 Phase 4：统一条件下比较微调前后表现**。先核对提示词、生成设置和评分规则，再形成可信的对比表；方案定稿后才使用测试集。

目前这些数字应统一称为**验证集结果**，还不能写成最终测试成绩。